In [1]:
import pandas as pd
from functools import partial

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
# dataset_root = "/home/y-guo/self-ensemble"

def get_ppl_filename(
        model_name, dataset_name, logits_ensemble_method, 
        is_baseline, repeat_paras, ensemble_method, 
        ensemble_layer, ensemble_alpha, token_mode, 
        multilayer, num_fewshots, num_paraphrases):
    
    if is_baseline:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.baseline."
    else:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.logits.{logits_ensemble_method}."
        if ensemble_method == "layer_output_avg":
            dump_file += f"avglayer.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        elif ensemble_method == "ffn_activation_max":
            dump_file += f"maxffn.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        if multilayer:
            dump_file += "multilayer."
    if num_fewshots != 5:
        dump_file += f"{num_fewshots}fshots."
    dump_file += f"{num_paraphrases}paras.feather"
    return dump_file

In [2]:
num_fewshots = 0
num_paraphrases = 5

get_ppl_filename_partial = partial(
    get_ppl_filename,
    logits_ensemble_method="avg",
    num_fewshots=num_fewshots,
    num_paraphrases=num_paraphrases,
    multilayer=True,
    repeat_paras=False,
    ensemble_method="layer_output_avg", 
    ensemble_alpha=1, 
    token_mode="last")

In [3]:
from notebooks._utils import calculate_accuracy, get_layers

# model_name = "llama3.2_3b"
for model_name in ["llama3.2_3b", "qwen2.5_3b", "qwen3_4b", "pythia_2.8b", "qwen3_30b"]:
    print(f"\n================ Evaluating model: {model_name} ================ ")
    for dataset in ["commonsense", "mmlu", "logiqa"]: 
    # for dataset in ["commonsense"]: 
        print(f"\n------ Evaluating dataset: {dataset} ------")
        basefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=True,
            ensemble_layer=get_layers(model_name))
        
        ensemblefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=False,
            ensemble_layer=get_layers(model_name))
        
        try:
            basedf = pd.read_feather(basefn)
            calculate_accuracy(basedf, label="Baseline", is_multichoice=True)
        except FileNotFoundError:
            print(f"Baseline file not found: {basefn}")

        try:
            ensembledf = pd.read_feather(ensemblefn)
            calculate_accuracy(ensembledf, label="Ensemble", is_multichoice=True)
        except FileNotFoundError:
            print(f"Ensemble file not found: {ensemblefn}")


================ Evaluating model: llama3.2_3b ================ 

------ Evaluating dataset: commonsense ------
Multichoice Acc: 0.4714 ==> 🏷️ Baseline
Multichoice Acc: 0.5000 ==> 🏷️ Ensemble

------ Evaluating dataset: mmlu ------
Baseline file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/mmlu_paraphrase/llama3.2_3b/mmluparaphrase.ppl.baseline.0fshots.5paras.feather
Ensemble file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/mmlu_paraphrase/llama3.2_3b/mmluparaphrase.ppl.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.5paras.feather

------ Evaluating dataset: logiqa ------
Baseline file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/logiqa_paraphrase/llama3.2_3b/logiqaparaphrase.ppl.baseline.0fshots.5paras.feather
Ensemble file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/logiqa_paraphrase/llama3.2_3b/logiqaparaphrase.ppl.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.5paras.feather

====

In [4]:
import os
os.path.exists(basefn)

False

In [5]:
basedf

,uuid,answers,prediction,generation,correctness,qonly_prompts,qchoice_prompts,paraphrases,ppls,best_choice_idx,choices_label,choices_text,answer_label,correct
0,000990552527b1353f98f1e1a7dfc643_1,E,E,E,None,[Answer the given question.\n\n\nQ: To what sy...,[Answer the given question.\n\n\nQ: To what sy...,[To what system did a star with four solid inn...,"[2839.988525390625, 1672.3316650390625, 218827...",solar system,"[A, B, C, D, E]","[hollywood, night sky, constellation, aliens, ...",E,1
1,000990552527b1353f98f1e1a7dfc643_1,E,E,E,None,[Answer the given question.\n\n\nQ: What name ...,[Answer the given question.\n\n\nQ: What name ...,[What name describes the collection that consi...,"[922.289794921875, 2283.260498046875, 121813.5...",solar system,"[A, B, C, D, E]","[hollywood, night sky, constellation, aliens, ...",E,1
2,000990552527b1353f98f1e1a7dfc643_1,E,E,E,None,[Answer the given question.\n\n\nQ: What large...,[Answer the given question.\n\n\nQ: What large...,[What larger structure was this star and its e...,"[6937.244140625, 207.2341766357422, 283.353210...",solar system,"[A, B, C, D, E]","[hollywood, night sky, constellation, aliens, ...",E,1
3,000990552527b1353f98f1e1a7dfc643_1,E,E,E,None,[Answer the given question.\n\n\nQ: This star ...,[Answer the given question.\n\n\nQ: This star ...,[This star had a total of eight planets—four t...,"[945.9764404296875, 261.6083984375, 1193.03430...",solar system,"[A, B, C, D, E]","[hollywood, night sky, constellation, aliens, ...",E,1
4,000990552527b1353f98f1e1a7dfc643_1,E,E,E,None,[Answer the given question.\n\n\nQ: Into what ...,[Answer the given question.\n\n\nQ: Into what ...,[Into what category of celestial systems would...,"[2266.99658203125, 2411.0966796875, 49328.3085...",solar system,"[A, B, C, D, E]","[hollywood, night sky, constellation, aliens, ...",E,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,ffb12644111a2dafcf8046dd9e3d90c7,A,A,A,None,[Answer the given question.\n\n\nQ: What categ...,[Answer the given question.\n\n\nQ: What categ...,[What category of housing would include sensor...,"[108.80570220947266, 445.4781799316406, 408.38...",modern house,"[A, B, C, D, E]","[modern house, building, townhouse, neighbor's...",A,1
4996,ffb12644111a2dafcf8046dd9e3d90c7,A,A,A,None,[Answer the given question.\n\n\nQ: Which kind...,[Answer the given question.\n\n\nQ: Which kind...,[Which kind of home is indicated by having lig...,"[41.843894958496094, 5869.86865234375, 125.249...",modern house,"[A, B, C, D, E]","[modern house, building, townhouse, neighbor's...",A,1
4997,ffb12644111a2dafcf8046dd9e3d90c7,A,A,A,None,[Answer the given question.\n\n\nQ: What sort ...,[Answer the given question.\n\n\nQ: What sort ...,[What sort of dwelling would have sensor-contr...,"[15.96840763092041, 1458.098876953125, 33.8122...",modern house,"[A, B, C, D, E]","[modern house, building, townhouse, neighbor's...",A,1
4998,ffb12644111a2dafcf8046dd9e3d90c7,A,A,A,None,[Answer the given question.\n\n\nQ: What kind ...,[Answer the given question.\n\n\nQ: What kind ...,[What kind of place to live would be character...,"[58.92100143432617, 423.63995361328125, 80.330...",modern house,"[A, B, C, D, E]","[modern house, building, townhouse, neighbor's...",A,1
